In [ ]:
import os
if os.path.exists('/content/checkmaize'):
    os.system('cd /content/checkmaize && git pull')
!pip install -q onnx onnxruntime -r /content/checkmaize/requirements.txt


In [ ]:
import os
os.chdir('/content/checkmaize')
winner = 'efficientnet_b0'
assert os.path.exists(f'artifacts/runs/{winner}/best.pt'), f'no checkpoint for {winner}'
print('winner:', winner)


In [ ]:
import os
os.chdir('/content/checkmaize')
!python -m inference.export --checkpoint artifacts/runs/{winner}/best.pt --out artifacts/runs/{winner}/model.onnx
!python -m inference.quantize --fp32 artifacts/runs/{winner}/model.onnx --out artifacts/model_int8.onnx
!python -m inference.verify --checkpoint artifacts/runs/{winner}/best.pt --fp32 artifacts/runs/{winner}/model.onnx --int8 artifacts/model_int8.onnx


In [ ]:
import csv, json, os
os.chdir('/content/checkmaize')
labels = ['common_rust', 'gray_leaf_spot', 'northern_leaf_blight', 'healthy']
with open('artifacts/labels.json', 'w') as f:
    json.dump(labels, f)
with open('artifacts/runs/' + winner + '/metrics.json') as f:
    m = json.load(f)
with open('benchmarks/report/comparison.csv') as f:
    row = [r for r in csv.DictReader(f) if r['model'] == winner][0]
with open('inference/verify_report.json') as f:
    v = json.load(f)
metrics = {
    'model': winner,
    'test_accuracy': m['accuracy'],
    'macro_f1': m['macro_f1'],
    'onnx_bytes': int(row['onnx_bytes']),
    'int8_test_accuracy': v['int8_accuracy'],
    'shipped': 'int8' if v['ship_int8'] else 'fp32',
}
with open('artifacts/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print(json.dumps(metrics, indent=2))


In [ ]:
import os, shutil
os.chdir('/content/checkmaize')
shutil.make_archive('/content/artifacts', 'zip', 'artifacts')
shutil.make_archive('/content/fixtures', 'zip', 'app/src/ml/__tests__/fixtures')
from google.colab import files
files.download('/content/artifacts.zip')
files.download('/content/fixtures.zip')
files.download('docs/onnx-contract.md')
